# CVE/CWE → Attack Family Mapping

Dataset: `CVE_CWE_2025.csv`

## 1. Import Libraries

Start by importing what we need: pandas for data handling, plus the CWE hierarchy file.

In [1]:
!pip install -q tf-keras
!pip install -q "transformers==4.40.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 3.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 58.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.4 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import transformers
from transformers import is_tf_available, TFBertModel

print(tf.__version__, transformers.__version__)
print(is_tf_available())

2.20.0 4.40.0
True


In [3]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [4]:
cwe_df = pd.read_csv("/kaggle/input/datasets/stanislavvinokur/cve-and-cwe-dataset-1999-2025/CVE_CWE_2025.csv")
cwe_df.head()

,ID,CVE-ID,CVSS-V4,CVSS-V3,CVSS-V2,SEVERITY,DESCRIPTION,CWE-ID
0,1,CVE-1999-0001,NaN,NaN,5.0,MEDIUM,ip_input.c in BSD-derived TCP/IP implementatio...,CWE-20
1,2,CVE-1999-0002,NaN,NaN,10.0,HIGH,Buffer overflow in NFS mountd gives root acces...,CWE-119
2,3,CVE-1999-0003,NaN,NaN,10.0,HIGH,Execute commands as root via buffer overflow i...,NVD-CWE-Other
3,4,CVE-1999-0004,NaN,NaN,5.0,MEDIUM,"MIME buffer overflow in email clients, e.g. So...",NVD-CWE-Other
4,5,CVE-1999-0005,NaN,NaN,10.0,HIGH,Arbitrary command execution via IMAP buffer ov...,NVD-CWE-Other


In [5]:
cwe_df["CWE-ID"].value_counts().head(30)

CWE-ID
NVD-CWE-Other    55550
CWE-79           35698
CWE-89           13925
CWE-119          12053
CWE-20           10851
CWE-787           9163
CWE-200           8587
CWE-352           7446
CWE-125           6786
CWE-22            6737
CWE-416           5397
CWE-264           5339
CWE-862           4441
CWE-94            4044
CWE-78            3959
CWE-476           3621
CWE-287           3360
CWE-284           3203
CWE-120           2950
CWE-434           2780
CWE-399           2637
CWE-190           2459
CWE-310           2297
CWE-400           2241
CWE-77            2162
CWE-74            1971
CWE-269           1967
CWE-121           1779
CWE-502           1707
CWE-863           1701
Name: count, dtype: int64

## 2. Build CWE/CVE → Attack Family Mapping

Attack family maps individual CWE-IDs into 13 broader attack families (Injection, XSS, 
Memory Corruption, etc.) based on the MITRE CWE hierarchy and frequency 
analysis of the top CWEs in our dataset.

In [6]:
attack_family = {
    # Injection-related
    "CWE-89": "Injection",        # SQL Injection
    "CWE-78": "Injection",        # OS Command Injection
    "CWE-77": "Injection",        # Command Injection (general)
    "CWE-94": "Injection",        # Code Injection
    "CWE-74": "Injection",        # Injection (general/base)
    "CWE-20": "Injection",        # Improper Input Validation (judgment call)
    "CWE-918": "Injection",       # SSRF
    "CWE-611": "Injection",       # XXE
    "CWE-427": "Injection",       # Uncontrolled Search Path Element
    "CWE-601": "Injection",      
    
    # Cross-Site Scripting
    "CWE-79": "XSS",

    # Memory Corruption
    "CWE-119": "Memory Corruption",  # Buffer overflow (general)
    "CWE-787": "Memory Corruption",  # Out-of-bounds Write
    "CWE-125": "Memory Corruption",  # Out-of-bounds Read
    "CWE-416": "Memory Corruption",  # Use After Free
    "CWE-476": "Memory Corruption",  # NULL Pointer Dereference
    "CWE-120": "Memory Corruption",  # Buffer Copy w/o Checking Size
    "CWE-190": "Memory Corruption",  # Integer Overflow
    "CWE-121": "Memory Corruption",  # Stack-based Buffer Overflow
    "CWE-122": "Memory Corruption",  # Heap-based Buffer Overflow
    "CWE-189": "Memory Corruption",  # Numeric Errors

    # Info Disclosure
    "CWE-200": "Info Disclosure",
    "CWE-532": "Info Disclosure",    # Insertion of Sensitive Info into Log Files

    # CSRF
    "CWE-352": "CSRF",

    # Path Traversal
    "CWE-22": "Path Traversal",
    "CWE-59": "Path Traversal",      # Improper Link Resolution (symlink attacks)

    # Authentication & Access Control
    "CWE-264": "Authentication & Access Control",  # Permissions/Privileges/Access Control
    "CWE-284": "Authentication & Access Control",  # Improper Access Control
    "CWE-862": "Authentication & Access Control",  # Missing Authorization
    "CWE-863": "Authentication & Access Control",  # Incorrect Authorization
    "CWE-269": "Authentication & Access Control",  # Improper Privilege Management
    "CWE-287": "Authentication & Access Control",  # Improper Authentication
    "CWE-306": "Authentication & Access Control",  # Missing Authentication for Critical Function
    "CWE-732": "Authentication & Access Control",  # Incorrect Permission Assignment
    "CWE-798": "Authentication & Access Control",  # Hard-coded Credentials
    "CWE-276": "Authentication & Access Control",  # Incorrect Default Permissions
    "CWE-522": "Authentication & Access Control",  # Insufficiently Protected Credentials
    "CWE-639": "Authentication & Access Control",  # Authorization Bypass via User-Controlled Key
    "CWE-255": "Authentication & Access Control",  # Credentials Management Errors

    # File Handling
    "CWE-434": "File Handling",  # Unrestricted Upload of File with Dangerous Type

    # Denial of Service
    "CWE-400": "Denial of Service",  # Uncontrolled Resource Consumption
    "CWE-399": "Denial of Service",  # Resource Management Errors
    "CWE-770": "Denial of Service",  # Allocation of Resources Without Limits
    "CWE-401": "Denial of Service",  # Missing Release of Memory (memory leak)

    # Cryptographic Issues
    "CWE-310": "Cryptographic Issues",
    "CWE-295": "Cryptographic Issues",  # Improper Certificate Validation

    # Deserialization
    "CWE-502": "Deserialization",

    # Race Conditions
    "CWE-362": "Race Condition",

    # NVD's catch-all bucket
    "NVD-CWE-Other": "Other",
}

## 3. Apply the Mapping

Apply the dictionary to create a new `attack_family` column, with any 
unmapped CWE falling back to "Other".

In [7]:
cwe_df["attack_family"] = cwe_df["CWE-ID"].map(attack_family).fillna("Other")
cwe_df["attack_family"].value_counts()

attack_family
Other                              88620
Memory Corruption                  46781
Injection                          41421
XSS                                35698
Authentication & Access Control    27247
Info Disclosure                     9377
Path Traversal                      7917
CSRF                                7446
Denial of Service                   6785
Cryptographic Issues                3341
File Handling                       2780
Deserialization                     1707
Race Condition                      1574
Name: count, dtype: int64

## 4. Checking some of the Labels

I realized that sometimes CWE labels may reflect the software weakness rather than the description, which can lead to inconsistencies between the description and the ID. So I was hoping the NLP would catch that and correct it.

In [8]:
cwe_df[["DESCRIPTION", "attack_family"]].sample(10)

,DESCRIPTION,attack_family
110033,This vulnerability allows remote attackers to ...,Memory Corruption
95229,The S/MIME specification allows a Cipher Block...,Other
46752,Cross-site scripting (XSS) vulnerability in th...,XSS
61363,Multiple integer signedness errors in libpurpl...,Memory Corruption
187116,vRealize Operations (vROps) contains a privile...,Authentication & Access Control
97862,Vulnerability in the Oracle FLEXCUBE Direct Ba...,Other
271124,NVIDIA CUDA Toolkit for all platforms contains...,Other
34646,PreProjects Pre Resume Submitter stores online...,Authentication & Access Control
35771,GNOME Rhythmbox 0.11.5 allows remote attackers...,Injection
219864,A cross-site request forgery (CSRF) vulnerabil...,CSRF


## 5. Define Features (X) and Target (y)

- **X**: the CVE description text (model input)
- **y**: the attack_family label (what we're predicting)

In [9]:
X = cwe_df["DESCRIPTION"]
y = cwe_df["attack_family"]

In [10]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Initialize the encoder
encoder = LabelEncoder()

# Transform ['red', 'blue', 'red'] into [1, 0, 1]
y_encoded = encoder.fit_transform(y).astype('int32')

# Save the number of classes for your model configuration
num_classes = len(encoder.classes_)
print(f"Total unique classes: {num_classes}")
print(f"Class mapping: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")


Total unique classes: 13
Class mapping: {'Authentication & Access Control': np.int64(0), 'CSRF': np.int64(1), 'Cryptographic Issues': np.int64(2), 'Denial of Service': np.int64(3), 'Deserialization': np.int64(4), 'File Handling': np.int64(5), 'Info Disclosure': np.int64(6), 'Injection': np.int64(7), 'Memory Corruption': np.int64(8), 'Other': np.int64(9), 'Path Traversal': np.int64(10), 'Race Condition': np.int64(11), 'XSS': np.int64(12)}


## 6. Train/Test Split

Split the data 80/20, stratified by `attack_family` to preserve class 
proportions across both sets since we have a class imbalance.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. Training BERT

In [12]:
mapped_only = cwe_df[cwe_df["attack_family"] != "Other"]
X_mapped = mapped_only["DESCRIPTION"]
y_mapped = mapped_only["attack_family"]

In [13]:
print(f"Rows before: {len(cwe_df)}")
print(f"Rows after excluding Other: {len(mapped_only)}")

Rows before: 280694
Rows after excluding Other: 192074


In [14]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mapped, y_mapped, test_size=0.2, random_state=42, stratify=y_mapped
)

In [15]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint

import transformers
from tqdm.notebook import tqdm
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizer

In [16]:
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')
def bert_encode(data, maximum_length, batch_size=256):
    # Convert input to a standard Python list
    text_list = data.tolist() if hasattr(data, 'tolist') else list(data)
    
    input_ids = []
    attention_masks = []
    
    # Loop through data in chunks instead of row-by-row
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        
        encoded = tokenizer(
            batch,  # Tokenizes the whole batch efficiently in parallel
            add_special_tokens=True,
            max_length=maximum_length,
            return_attention_mask=True,
            padding='max_length',
            truncation=True,
        )
        input_ids.extend(encoded['input_ids'])
        attention_masks.extend(encoded['attention_mask'])
        
    return np.array(input_ids), np.array(attention_masks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

In [17]:
train_input_ids,train_attention_masks = bert_encode(X,100)

In [18]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

def create_model(bert_model, maximum_length,num_classes):
    # 1. Define symbolic input shapes (Placeholders for data)
    input_ids = tf.keras.layers.Input(shape=(maximum_length,), dtype=tf.int32, name="input_ids")
    attention_masks = tf.keras.layers.Input(shape=(maximum_length,), dtype=tf.int32, name="attention_masks")
    
    # 2. Extract BERT embeddings symbolically
    # Note: Use transformer model output dictionary/object correctly
    bert_outputs = bert_model([input_ids, attention_masks])
    
    # Extract pooler_output (index 1 or attribute) for classification
    if hasattr(bert_outputs, 'pooler_output'):
        output = bert_outputs.pooler_output
    else:
        output = bert_outputs[1]
        
    # 3. Add your custom classification layers
    output = tf.keras.layers.Dense(32, activation='relu')(output)
    output = tf.keras.layers.Dropout(0.2)(output)
    output = tf.keras.layers.Dense(num_classes, activation='softmax')(output)

    # 4. Construct and compile the model skeleton
    model = tf.keras.models.Model(inputs=[input_ids, attention_masks], outputs=output)
    
    # Bug Fix: learning_rate instead of lr (lr was deprecated and removed)
    model.compile(optimizer=Adam(learning_rate=1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model

In [19]:
bert_model = TFBertModel.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

I0000 00:00:1784684122.713272      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784684122.719263      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPre

In [20]:
model = create_model(bert_model,100,num_classes)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 100)]                0         []                            
                                                                                                  
 attention_masks (InputLaye  [(None, 100)]                0         []                            
 r)                                                                                               
                                                                                                  
 tf_bert_model (TFBertModel  TFBaseModelOutputWithPooli   1094822   ['input_ids[0][0]',           
 )                           ngAndCrossAttentions(last_   40         'attention_masks[0][0]']     
                             hidden_state=(None, 100, 7                                       

In [ ]:
history = model.fit(
    [train_input_ids,train_attention_masks],
    y_encoded,
    validation_split=0.2,
    epochs=3,
    batch_size=17
)

Epoch 1/3
 9771/13210 [=====================>........] - ETA: 22:14 - loss: 0.8223 - accuracy: 0.7171

In [ ]:
from sklearn.metrics import classification_report,accuracy_score
val_input_ids,val_attention_masks = bert_encode(X_test_m,100)
y_pred = model.predict(
    [val_input_ids,val_attention_masks],
    batch_size=16,
)
y_pred_labels = np.argmax(y_pred,axis=1)

In [ ]:
encoder = LabelEncoder()

# Transform ['red', 'blue', 'red'] into [1, 0, 1]
y_encoded_test = encoder.fit_transform(y_test_m).astype('int32')

# Save the number of classes for your model configuration
num_classes = len(encoder.classes_)
print(f"Total unique classes: {num_classes}")
print(f"Class mapping: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

In [ ]:
print("==========Classification Report================")
print(classification_report(
    y_true=y_encoded_test,
    y_pred=y_pred_labels,
    target_names=encoder.classes_
))
overall_accuracy = accuracy_score(y_encoded_test,y_pred_labels)
print(f"Overall Accuracy {overall_accuracy:.4f}")

## Summary

I tested a few variations before settling on this model on step 9:
- Baseline (unigrams, no class weighting): 0.74 accuracy
- Balanced class weights: 0.68 accuracy (higher recall on rare classes, 
  but much lower precision)
- Balanced + bigrams: 0.71 accuracy
- Bigrams only (final, above): 0.77 accuracy
- 12-class (Other excluded, step 11): 0.89 accuracy, 0.84 macro F1